# Homework 5

# Задача №1 - Можете ли вы отличить сорняки от рассады?

Теперь приступим к задаче классификации на картинках. Реализуйте программу, которая определяет тип рассады на изображении. 

Для того, чтобы определить характерные особенности каждого типа рассады, у вас есть train. Train это папка, в которой картинки уже классифицированы и лежат в соответствующих папках. Исходя из этой информации можете найти признаки, присущие конкретному растению.

Проверка вашего решения будет на происходить на test. В папке test уже нет метки класса для каждой картинки. 

[Ссылка на Яндекс-диск](https://yadi.sk/d/0Zzp0klXT0iRmA), все картинки тут.

Примеры изображений для теста:
<table><tr>
    <td> <img src="https://i.ibb.co/tbqR37m/fhj.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/6yL3Wmt/sfg.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/pvn7NvF/asd.png" alt="Drawing" style="width: 200px;"/> </td>
</tr></table>

In [78]:
import os
import cv2
import numpy as np
from glob import glob
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from typing import List, Dict, Tuple, Any, Optional
from sklearn.metrics import accuracy_score
from tqdm import tqdm

In [79]:
def extract_sift_features(img_path: str) -> Optional[np.ndarray]:
    image = cv2.imread(img_path, cv2.IMREAD_COLOR)

    if image is None:
        return None

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    color_mask = cv2.inRange(hsv, (35, 50, 50), (85, 255, 255))

    gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    masked_gray = cv2.bitwise_and(gray, gray, mask=color_mask)

    sift = cv2.SIFT_create()
    _, desc = sift.detectAndCompute(masked_gray, None)
    return desc


def build_visual_vocabulary(
    image_list: List[str], num_clusters: int, seed: int = 42
) -> KMeans:
    descriptors_collection = []

    for img_file in tqdm(image_list, desc="Extracting SIFT descriptors"):
        desc = extract_sift_features(img_file)
        if desc is not None:
            descriptors_collection.append(desc)
    print()

    all_descriptors = np.vstack(descriptors_collection)

    kmeans_model = KMeans(n_clusters=num_clusters, random_state=seed)
    kmeans_model.fit(all_descriptors)

    return kmeans_model


def compute_feature_vector(
    image_path: str, kmeans_model: KMeans, num_clusters: int
) -> np.ndarray:
    desc = extract_sift_features(image_path)

    if desc is None:
        return np.zeros(num_clusters, dtype=np.float32)

    cluster_indices = kmeans_model.predict(desc)

    histogram, _ = np.histogram(
        cluster_indices, bins=num_clusters, range=(0, num_clusters)
    )
    histogram = histogram.astype(np.float32)

    if histogram.sum() > 0:
        histogram /= histogram.sum()

    return histogram


def load_data(dataset_path: str) -> Tuple[List[str], List[int], List[str]]:
    categories = sorted(os.listdir(dataset_path))
    image_paths = []
    labels = []

    for idx, category in enumerate(categories):
        category_dir = os.path.join(dataset_path, category)
        for img_path in glob(os.path.join(category_dir, "*.png")):
            image_paths.append(img_path)
            labels.append(idx)

    return image_paths, labels, categories


def train_plant_model(
    train_folder: str, cluster_count: int = 100, random_seed: int = 42
) -> Tuple[Pipeline, KMeans, List[str]]:
    print("Loading training images...")
    print()
    train_files, train_labels, classes = load_data(train_folder)

    print("Creating visual vocabulary...")
    print()
    kmeans = build_visual_vocabulary(train_files, cluster_count, random_seed)

    print("Computing feature vectors for training images...")
    print()

    features = []
    for file in tqdm(train_files, desc="Processing training images"):
        fv = compute_feature_vector(file, kmeans, cluster_count)
        features.append(fv)
    print()

    X_train = np.array(features)
    y_train = np.array(train_labels)

    print("Fitting classifier...")
    print()
    clf = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("logreg", LogisticRegression(random_state=random_seed)),
        ]
    )
    clf.fit(X_train, y_train)

    print("Training completed!")
    print()
    return clf, kmeans, classes


def evaluate_plant_model(
    test_folder: str,
    model: Pipeline,
    kmeans: KMeans,
    classes: List[str],
    cluster_count: int,
) -> Tuple[List[Dict[str, Any]], List[int]]:
    test_files = glob(os.path.join(test_folder, "*.png"))
    evaluation_results = []
    predicted_labels = []

    print(f"Evaluating on {len(test_files)} test images...")
    print()

    for file in tqdm(test_files, desc="Processing test images"):
        fv = compute_feature_vector(file, kmeans, cluster_count)
        probabilities = model.predict_proba([fv])[0]

        best_idx = np.argmax(probabilities)
        best_class = classes[best_idx]
        confidence = probabilities[best_idx]

        predicted_labels.append(best_idx)
        evaluation_results.append(
            {
                "image_path": file,
                "predictions": list(zip(classes, probabilities)),
                "top_class": best_class,
                "top_class_idx": best_idx,
                "confidence": confidence,
            }
        )

    return evaluation_results, predicted_labels

In [80]:
train_dir = os.path.join("plants", "train")
test_dir = os.path.join("plants", "test")

feature_size = 100
random_state = 42

y_test = np.array(
    [
        0,
        0,
        1,
        1,
        0,
        0,
        2,
        3,
        1,
        0,
        1,
        1,
        0,
        3,
        2,
        3,
        2,
        3,
        0,
        3,
        2,
        2,
        3,
        1,
        3,
        2,
        2,
        1,
        3,
        3,
        0,
        2,
        0,
        0,
        1,
        2,
        3,
        2,
        1,
        1,
    ]
)

print()
clf, kmeans, classes = train_plant_model(
    train_dir, feature_size, random_state
)


Loading training images...

Creating visual vocabulary...



Extracting SIFT descriptors: 100%|██████████| 20/20 [00:02<00:00,  8.76it/s]



Computing feature vectors for training images...



Processing training images: 100%|██████████| 20/20 [00:02<00:00,  8.69it/s]


Fitting classifier...

Training completed!



In [81]:
print()
test_results, y_pred = evaluate_plant_model(
    test_dir, clf, kmeans, classes, feature_size
)
print()

print("Summary metrics:")
print()

print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print()


Evaluating on 40 test images...



Processing test images: 100%|██████████| 40/40 [00:05<00:00,  7.92it/s]


Summary metrics:

Accuracy: 1.0



# Задача №2 - Собери пазл (2.0).

Даны кусочки изображения, ваша задача склеить пазл в исходную картинку. 

Условия:
* Дано исходное изображение для проверки, использовать собранное изображение в самом алгоритме нельзя;
* Картинки имеют друг с другом пересечение;
* После разрезки кусочки пазлов не были повернуты или отражены;
* НЕЛЬЗЯ выбрать опорную картинку для сбора пазла, как это было в homework 3
* В процессе проверки решения пазлы могут быть перемешаны, т.е. порядок пазлов в проверке может отличаться от исходного 

Изображения расположены по [ссылке](https://disk.yandex.ru/d/XtpawH1sV9UDlg).

Примеры изображений:
<img src="puzzle/su_fighter.jpg" alt="Drawing" style="width: 300px;"/>
<table><tr>
    <td> <img src="puzzle/su_fighter_shuffle/0.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/1.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/2.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/3.jpg" alt="Drawing" style="width: 200px;"/> </td>
</tr></table>

In [82]:
# Ваш код